In [ ]:
import numpy as np
import pandas as pd
import os
import gc

In [2]:
# ---- LOAD DATASET ----
parquet_path = r"D:\\Important\\Semester\\Semester X\\MTP\\LSI\\shimla_dataset.parquet"

print("Loading dataset...")
df = pd.read_parquet(parquet_path)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Memory: {df.memory_usage(deep=True).sum() / (1024**3):.2f} GB")

Loading dataset...
Shape: (51423700, 15)
Columns: ['pixel_row', 'pixel_col', 'landslide_dist', 'slope', 'lulc', 'elevation', 'road_dist', 'builtup_dist', 'land_value', 'water_dist', 'lineament_dist', 'lst', 'railway_dist', 'lsi_continuous', 'lsi_classified']
Memory: 2.73 GB


In [3]:
# ---- QUICK VERIFICATION ----
print(f"\nNaN check: {df.isna().sum().sum()}")
print(f"LSI range: {df['lsi_continuous'].min():.4f} to {df['lsi_continuous'].max():.4f}")
print(f"Coordinate ranges: row [{df['pixel_row'].min()}-{df['pixel_row'].max()}], col [{df['pixel_col'].min()}-{df['pixel_col'].max()}]")

# ---- CLASS DISTRIBUTION ----
print(f"\nClass distribution:")
class_labels = {1: "Very Low", 2: "Low", 3: "Moderate", 4: "High", 5: "Very High"}
counts = df['lsi_classified'].value_counts().sort_index()
for cls, count in counts.items():
    pct = (count / len(df)) * 100
    print(f"  {int(cls)} ({class_labels[int(cls)]:<10}): {count:>12,} ({pct:.2f}%)")


NaN check: 0
LSI range: 2.0400 to 5.0000
Coordinate ranges: row [2-10551], col [1-12543]

Class distribution:
  1 (Very Low  ):    1,060,985 (2.06%)
  2 (Low       ):   13,055,064 (25.39%)
  3 (Moderate  ):   31,455,011 (61.17%)
  4 (High      ):    5,826,106 (11.33%)
  5 (Very High ):       26,534 (0.05%)


In [4]:
# ---- FIX LULC DTYPE ----
df["lulc"] = df["lulc"].astype(np.int8)
print(f"LULC dtype: {df['lulc'].dtype}")
print(f"LULC unique values: {sorted(df['lulc'].unique())}")

LULC dtype: int8
LULC unique values: [np.int8(1), np.int8(2), np.int8(5), np.int8(7), np.int8(8), np.int8(9), np.int8(11)]


In [5]:
# ---- ASSIGN SPATIAL BLOCKS ----
block_size_px = 100  # 100 pixels = 1 km

df["block_row"] = (df["pixel_row"] // block_size_px).astype(np.int16)
df["block_col"] = (df["pixel_col"] // block_size_px).astype(np.int16)

# Create unique block ID from row and col
df["block_id"] = df["block_row"] * 1000 + df["block_col"]

# ---- BLOCK STATISTICS ----
total_blocks = df["block_id"].nunique()
pixels_per_block = df.groupby("block_id").size()

print(f"Total blocks with valid pixels: {total_blocks}")
print(f"\nPixels per block:")
print(f"  Min: {pixels_per_block.min():,}")
print(f"  Max: {pixels_per_block.max():,}")
print(f"  Mean: {pixels_per_block.mean():,.0f}")
print(f"  Median: {pixels_per_block.median():,.0f}")

# ---- CHECK CLASS PRESENCE IN BLOCKS ----
print(f"\nBlocks containing each class:")
for cls in sorted(df["lsi_classified"].unique()):
    label = class_labels[int(cls)]
    blocks_with_class = df[df["lsi_classified"] == cls]["block_id"].nunique()
    pct = (blocks_with_class / total_blocks) * 100
    print(f"  Class {int(cls)} ({label:<10}): {blocks_with_class} blocks ({pct:.1f}%)")

Total blocks with valid pixels: 5464

Pixels per block:
  Min: 1
  Max: 10,000
  Mean: 9,411
  Median: 10,000

Blocks containing each class:
  Class 1 (Very Low  ): 2083 blocks (38.1%)
  Class 2 (Low       ): 5158 blocks (94.4%)
  Class 3 (Moderate  ): 5351 blocks (97.9%)
  Class 4 (High      ): 3433 blocks (62.8%)
  Class 5 (Very High ): 143 blocks (2.6%)


In [7]:
from sklearn.model_selection import train_test_split

In [8]:
np.random.seed(42)

# ---- IDENTIFY BLOCKS WITH CLASS 5 ----
class5_blocks = df[df["lsi_classified"] == 5]["block_id"].unique()
other_blocks = df[~df["block_id"].isin(class5_blocks)]["block_id"].unique()

print(f"Blocks with Class 5: {len(class5_blocks)}")
print(f"Blocks without Class 5: {len(other_blocks)}")


Blocks with Class 5: 143
Blocks without Class 5: 5321


In [9]:
# ---- SPLIT CLASS 5 BLOCKS (70/15/15) ----
c5_train, c5_temp = train_test_split(class5_blocks, test_size=0.30, random_state=42)
c5_val, c5_test = train_test_split(c5_temp, test_size=0.50, random_state=42)

# ---- SPLIT REMAINING BLOCKS (70/15/15) ----
other_train, other_temp = train_test_split(other_blocks, test_size=0.30, random_state=42)
other_val, other_test = train_test_split(other_temp, test_size=0.50, random_state=42)

# ---- COMBINE ----
train_blocks = np.concatenate([c5_train, other_train])
val_blocks = np.concatenate([c5_val, other_val])
test_blocks = np.concatenate([c5_test, other_test])

print(f"\nBlock split:")
print(f"  Train: {len(train_blocks)} blocks ({len(train_blocks)/5464*100:.1f}%)")
print(f"  Val:   {len(val_blocks)} blocks ({len(val_blocks)/5464*100:.1f}%)")
print(f"  Test:  {len(test_blocks)} blocks ({len(test_blocks)/5464*100:.1f}%)")


Block split:
  Train: 3824 blocks (70.0%)
  Val:   819 blocks (15.0%)
  Test:  821 blocks (15.0%)


In [10]:
# ---- ASSIGN SPLIT LABELS TO DATAFRAME ----
df["split"] = "unassigned"
df.loc[df["block_id"].isin(train_blocks), "split"] = "train"
df.loc[df["block_id"].isin(val_blocks), "split"] = "val"
df.loc[df["block_id"].isin(test_blocks), "split"] = "test"

# ---- VERIFY PIXEL COUNTS ----
split_counts = df["split"].value_counts()
print(f"\nPixel counts per split:")
for split_name in ["train", "val", "test"]:
    count = split_counts[split_name]
    pct = (count / len(df)) * 100
    print(f"  {split_name}: {count:>12,} ({pct:.1f}%)")

# ---- CLASS DISTRIBUTION PER SPLIT ----
print(f"\nClass distribution per split:")
print(f"  {'Class':<12} {'Train':>12} {'Val':>12} {'Test':>12}")
print("  " + "-" * 50)
for cls in sorted(df["lsi_classified"].unique()):
    label = class_labels[int(cls)]
    train_c = len(df[(df["split"] == "train") & (df["lsi_classified"] == cls)])
    val_c = len(df[(df["split"] == "val") & (df["lsi_classified"] == cls)])
    test_c = len(df[(df["split"] == "test") & (df["lsi_classified"] == cls)])
    print(f"  {int(cls)} {label:<10} {train_c:>12,} {val_c:>12,} {test_c:>12,}")


Pixel counts per split:
  train:   35,978,289 (70.0%)
  val:    7,697,143 (15.0%)
  test:    7,748,268 (15.1%)

Class distribution per split:
  Class               Train          Val         Test
  --------------------------------------------------
  1 Very Low        734,446      168,993      157,546
  2 Low           9,222,182    1,912,235    1,920,647
  3 Moderate     21,910,197    4,789,846    4,754,968
  4 High          4,094,576      823,874      907,656
  5 Very High        16,888        2,195        7,451


In [14]:
# ---- SUBSAMPLING CONFIGURATION ----
target_train = 700_000
target_val = 150_000
target_test = 150_000

np.random.seed(42)

def stratified_subsample(df_split, target_size, split_name):
    """Subsample while keeping ALL Class 5 and proportional others."""
    
    # Separate Class 5 (keep all) from others
    class5 = df_split[df_split["lsi_classified"] == 5]
    others = df_split[df_split["lsi_classified"] != 5]
    
    # Remaining budget after keeping all Class 5
    remaining_budget = target_size - len(class5)
    
    # Proportional sampling from other classes
    sampled_others = []
    other_classes = sorted(others["lsi_classified"].unique())
    
    for cls in other_classes:
        cls_data = others[others["lsi_classified"] == cls]
        # Proportion of this class among non-Class5 pixels
        proportion = len(cls_data) / len(others)
        n_samples = int(remaining_budget * proportion)
        # Don't sample more than available
        n_samples = min(n_samples, len(cls_data))
        sampled = cls_data.sample(n=n_samples, random_state=42)
        sampled_others.append(sampled)
    
    # Combine
    result = pd.concat([class5] + sampled_others, ignore_index=True)
    
    # Report
    print(f"\n  {split_name}: {len(df_split):,} → {len(result):,}")
    for cls in sorted(result["lsi_classified"].unique()):
        label = class_labels[int(cls)]
        count = len(result[result["lsi_classified"] == cls])
        pct = (count / len(result)) * 100
        print(f"    Class {int(cls)} ({label:<10}): {count:>8,} ({pct:.2f}%)")
    
    return result

In [15]:
# ---- SUBSAMPLE EACH SPLIT ----
print("=" * 50)
print("STRATIFIED SUBSAMPLING")
print("=" * 50)

train_sub = stratified_subsample(df[df["split"] == "train"], target_train, "TRAIN")
val_sub = stratified_subsample(df[df["split"] == "val"], target_val, "VAL")
test_sub = stratified_subsample(df[df["split"] == "test"], target_test, "TEST")

# ---- COMBINE INTO SUBSAMPLED DATASET ----
train_sub["split"] = "train"
val_sub["split"] = "val"
test_sub["split"] = "test"

df_subsampled = pd.concat([train_sub, val_sub, test_sub], ignore_index=True)

print(f"\n{'=' * 50}")
print(f"SUBSAMPLED DATASET SUMMARY")
print(f"{'=' * 50}")
print(f"Total: {len(df_subsampled):,}")
print(f"  Train: {len(train_sub):,}")
print(f"  Val:   {len(val_sub):,}")
print(f"  Test:  {len(test_sub):,}")
print(f"Memory: {df_subsampled.memory_usage(deep=True).sum() / (1024**2):.1f} MB")

# ---- CLEANUP ----
del train_sub, val_sub, test_sub
gc.collect()

STRATIFIED SUBSAMPLING

  TRAIN: 35,978,289 → 699,998
    Class 1 (Very Low  ):   13,951 (1.99%)
    Class 2 (Low       ):  175,181 (25.03%)
    Class 3 (Moderate  ):  416,199 (59.46%)
    Class 4 (High      ):   77,779 (11.11%)
    Class 5 (Very High ):   16,888 (2.41%)

  VAL: 7,697,143 → 149,999
    Class 1 (Very Low  ):    3,246 (2.16%)
    Class 2 (Low       ):   36,730 (24.49%)
    Class 3 (Moderate  ):   92,003 (61.34%)
    Class 4 (High      ):   15,825 (10.55%)
    Class 5 (Very High ):    2,195 (1.46%)

  TEST: 7,748,268 → 149,998
    Class 1 (Very Low  ):    2,901 (1.93%)
    Class 2 (Low       ):   35,369 (23.58%)
    Class 3 (Moderate  ):   87,563 (58.38%)
    Class 4 (High      ):   16,714 (11.14%)
    Class 5 (Very High ):    7,451 (4.97%)

SUBSAMPLED DATASET SUMMARY
Total: 999,995
  Train: 699,998
  Val:   149,999
  Test:  149,998
Memory: 72.2 MB


0

In [17]:
# ---- SAVE SUBSAMPLED SPATIAL DATASET ----
sub_path = r"D:\\Important\\Semester\\Semester X\\MTP\\LSI\\shimla_subsampled_spatial.parquet"
df_subsampled.to_parquet(sub_path, index=False, engine="pyarrow")
print(f"Subsampled spatial: {os.path.getsize(sub_path) / (1024**2):.1f} MB")

Subsampled spatial: 41.8 MB


In [18]:
# ---- CREATE RANDOM SPLIT DATASET (for comparison) ----
print("\nCreating random split dataset...")
np.random.seed(42)

# Subsample 1M from full dataset (no spatial awareness)
df_random = df.sample(n=1_000_000, random_state=42).copy()

rand_train, rand_temp = train_test_split(
    df_random, test_size=0.30, stratify=df_random["lsi_classified"], random_state=42
)
rand_val, rand_test = train_test_split(
    rand_temp, test_size=0.50, stratify=rand_temp["lsi_classified"], random_state=42
)

rand_train["split"] = "train"
rand_val["split"] = "val"
rand_test["split"] = "test"

df_random_split = pd.concat([rand_train, rand_val, rand_test], ignore_index=True)

# Save random split
rand_path = r"D:\\Important\\Semester\\Semester X\\MTP\\LSI\\shimla_subsampled_random.parquet"
df_random_split.to_parquet(rand_path, index=False, engine="pyarrow")
print(f"Subsampled random:  {os.path.getsize(rand_path) / (1024**2):.1f} MB")

# Report random split distribution
print(f"\nRandom split class distribution:")
print(f"  {'Class':<12} {'Train':>8} {'Val':>8} {'Test':>8}")
print("  " + "-" * 38)
for cls in sorted(df_random_split["lsi_classified"].unique()):
    label = class_labels[int(cls)]
    t = len(df_random_split[(df_random_split["split"] == "train") & (df_random_split["lsi_classified"] == cls)])
    v = len(df_random_split[(df_random_split["split"] == "val") & (df_random_split["lsi_classified"] == cls)])
    te = len(df_random_split[(df_random_split["split"] == "test") & (df_random_split["lsi_classified"] == cls)])
    print(f"  {int(cls)} {label:<10} {t:>8,} {v:>8,} {te:>8,}")

# ---- SUMMARY ----
print(f"\n{'=' * 50}")
print(f"ALL DATASETS SAVED")
print(f"{'=' * 50}")
print(f"1. Full spatial (already saved):     shimla_dataset.parquet ({1.75:.2f} GB)")
print(f"2. Subsampled spatial:               shimla_subsampled_spatial.parquet")
print(f"3. Subsampled random:                shimla_subsampled_random.parquet")

del df_random, rand_train, rand_val, rand_test, df_random_split
gc.collect()


Creating random split dataset...
Subsampled random:  42.4 MB

Random split class distribution:
  Class           Train      Val     Test
  --------------------------------------
  1 Very Low     14,430    3,092    3,092
  2 Low         177,172   37,965   37,966
  3 Moderate    428,485   91,818   91,818
  4 High         79,532   17,043   17,042
  5 Very High       381       82       82

ALL DATASETS SAVED
1. Full spatial (already saved):     shimla_dataset.parquet (1.75 GB)
2. Subsampled spatial:               shimla_subsampled_spatial.parquet
3. Subsampled random:                shimla_subsampled_random.parquet


0